In [33]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, TrainerCallback
import torch
from huggingface_hub import login
import os
from datasets import load_from_disk

In [34]:
# Set path for tokenized dataset
tokenized_path = "tokenized_tiny_codes_qwen2.5_0.5b"
os.environ["HF_DATASETS_CACHE"] = "E:"
torch.cuda.max_memory_allocated()

8237614080

In [35]:
# login into Hugging Face Hub
login()

In [36]:
# Load model and tokenizer
model_name = "Qwen/Qwen2-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.gradient_checkpointing_enable()

In [37]:
# Check if tokenized dataset exists
if os.path.exists(f"{tokenized_path}_train") and os.path.exists(f"{tokenized_path}_val"):
    print(f"Loading tokenized dataset from {tokenized_path}...")
    tokenized_ds = load_from_disk(f"{tokenized_path}_train")
    tokenized_val_ds = load_from_disk(f"{tokenized_path}_val")
else:
    print("Loading and tokenizing dataset...")
    ds = load_dataset("nampdn-ai/tiny-codes", split="train")

    # Split dataset into train and validation
    ds = ds.train_test_split(test_size=0.1) # 90% train, 10% validation
    train_ds = ds['train']
    val_ds = ds['test']

    def preprocess(batch):
        prompts = batch["prompt"]
        responses = batch["response"]
        results = { "input_ids": [], "attention_mask": [], "labels": [] }
        for prompt, response in zip(prompts, responses):
            messages = [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": response},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            tokenized = tokenizer(
                text,
                max_length=1024,
                truncation=True,
                padding="max_length"
            )
            results["input_ids"].append(tokenized["input_ids"])
            results["attention_mask"].append(tokenized["attention_mask"])
            results["labels"].append(tokenized["input_ids"].copy())
        return results

    tokenized_train_ds = train_ds.map(
        preprocess,
        batched=True,
        batch_size=512,
        remove_columns=train_ds.column_names,
        desc="Tokenizing training dataset"
    )

    tokenized_val_ds = val_ds.map(
        preprocess,
        batched=True,
        batch_size=512,
        remove_columns=val_ds.column_names,
        desc="Tokenizing validation dataset"
    )

    # Save both datasets
    tokenized_train_ds.save_to_disk(f"{tokenized_path}_train")
    tokenized_val_ds.save_to_disk(f"{tokenized_path}_val")
    tokenized_ds = tokenized_train_ds

Loading tokenized dataset from tokenized_tiny_codes_qwen2.5_0.5b...


Loading dataset from disk:   0%|          | 0/40 [00:00<?, ?it/s]

In [38]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=1e-5,
    logging_steps=100,
    save_steps=500,
    eval_steps=500,
    gradient_checkpointing=True,
    report_to=[],
    bf16=True,
)

In [39]:
class MemoryManagementCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    eval_dataset=tokenized_val_ds if 'tokenized_val_ds' in locals() else None,
    tokenizer=tokenizer,
    callbacks=[MemoryManagementCallback()]
)

# Train
trainer.train()

C:\Users\antho\AppData\Local\Temp\ipykernel_1828\2759779446.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
100,0.603300
200,0.425300
